In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Project Paths
# Purpose: Define the project root directory.
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM"
)

print(PROJECT_ROOT)
print(PROJECT_ROOT.exists())

/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM
False


In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Training Configuration
# Purpose: Define the training configuration for QLoRA fine-tuning.
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

OUTPUT_DIR = PROJECT_ROOT / "models" / "sft_qwen"

TRAIN_DATASET = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "train_sft.jsonl"
)

VALIDATION_DATASET = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "validation_sft.jsonl"
)

MAX_SEQ_LENGTH = 2048

NUM_EPOCHS = 3

LEARNING_RATE = 2e-4

BATCH_SIZE = 2

GRADIENT_ACCUMULATION = 4

RANDOM_STATE = 42

In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Install Required Libraries
# Purpose: Install stable dependencies for SFT with QLoRA.
# ============================================================

!pip install -q \
transformers==4.51.3 \
datasets==2.21.0 \
accelerate==1.5.2 \
trl==0.15.2 \
peft==0.15.2 \
bitsandbytes==0.46.0 \
pyarrow==16.1.0 \
sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fs

In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Mount Google Drive
# Purpose: Mount Google Drive to access the project files.
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from datasets import load_dataset

In [ ]:
train_dataset = load_dataset(
    "json",
    data_files=str(TRAIN_DATASET),
    split="train",
)

validation_dataset = load_dataset(
    "json",
    data_files=str(VALIDATION_DATASET),
    split="train",
)

print(train_dataset)
print()
print(validation_dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 92
})

Dataset({
    features: ['messages'],
    num_rows: 12
})


In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Import Training Libraries
# Purpose: Import all libraries required for QLoRA fine-tuning.
# ============================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
)

from trl import (
    SFTTrainer,
    SFTConfig,
)

In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Configure 4-bit Quantization
# Purpose: Configure BitsAndBytes for QLoRA training.
# ============================================================

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True,

)

print(bnb_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Load Tokenizer
# Purpose: Load the tokenizer associated with the base model.
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded.


In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Load Base Model
# Purpose: Load the quantized Qwen model for QLoRA fine-tuning.
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded successfully.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Configure LoRA
# Purpose: Prepare the model for parameter-efficient fine-tuning.
# ============================================================

from peft import (
    LoraConfig,
    get_peft_model,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
print("model" in globals())
print("tokenizer" in globals())

True
True


In [ ]:
import transformers
import trl
import peft
import datasets
import accelerate
import bitsandbytes

print(transformers.__version__)
print(trl.__version__)
print(peft.__version__)
print(datasets.__version__)
print(accelerate.__version__)
print(bitsandbytes.__version__)

4.51.3
0.15.2
0.15.2
2.21.0
1.5.2
0.46.0


In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Configure SFT Training
# Purpose: Define the training configuration.
# ============================================================

from trl import SFTConfig

training_args = SFTConfig(

    output_dir=str(OUTPUT_DIR),

    num_train_epochs=NUM_EPOCHS,

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRADIENT_ACCUMULATION,

    logging_steps=5,

    save_strategy="epoch",

    eval_strategy="epoch",

    save_total_limit=2,

    load_best_model_at_end=True,

    fp16=True,

    report_to="none",

    seed=RANDOM_STATE,

)

In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Create SFT Trainer
# Purpose: Initialize the trainer.
# ============================================================

trainer = SFTTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=validation_dataset,

    processing_class=tokenizer,

)

print("SFT Trainer created successfully.")

Converting train dataset to ChatML:   0%|          | 0/92 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/92 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/12 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


SFT Trainer created successfully.


In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Train Model
# Purpose: Run supervised fine-tuning.
# ============================================================

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.049300,0.278442
2,0.002600,0.001964


TrainOutput(global_step=33, training_loss=0.5026403093484767, metrics={'train_runtime': 185.8122, 'train_samples_per_second': 1.485, 'train_steps_per_second': 0.178, 'total_flos': 2090030138916864.0, 'train_loss': 0.5026403093484767})

In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Save LoRA Adapter
# Purpose: Save the fine-tuned adapter.
# ============================================================

trainer.save_model(
    OUTPUT_DIR
)

tokenizer.save_pretrained(
    OUTPUT_DIR
)

print("LoRA adapter saved successfully.")

LoRA adapter saved successfully.


In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 02_sft.ipynb
# Section: Verify Saved Model
# Purpose: Display saved files.
# ============================================================

import os

print(os.listdir(OUTPUT_DIR))

['checkpoint-24', 'checkpoint-33', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'training_args.bin']
